# Calidad de datos y exploración del proyecto integrador
Comprensión y preparación de datos de CRISP-DM. Ejecutar antes el pipeline de `docs/DATOS.md` y seleccionar el entorno Python del proyecto. Este notebook consume las salidas y no duplica las reglas de limpieza. No entrena modelos.
La exploración de patrones se limita al conjunto sugerido de entrenamiento 2021–2023. Los cortes deben acordarse con el equipo antes de comparar modelos.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
cwd = Path.cwd()
backend = next(p for p in [cwd, cwd / 'backend', *cwd.parents] if (p / 'data_pipeline').is_dir())
reports = backend.parent / 'docs' / 'eda'
audit = json.loads((reports / 'auditoria.json').read_text(encoding='utf-8'))
display(audit['fuentes'], audit['accidentes'], audit['poblacion'])

## Verificar la conciliación
Las exclusiones son disjuntas. Una comuna rural queda fuera por alcance; una comuna desconocida es un problema diferente. Las coordenadas no son necesarias para contar cuando la comuna ya está identificada. Los registros sin coordenadas permanecen en el conteo.

In [ ]:
control = audit['accidentes']
assert sum(control['exclusiones_disjuntas'].values()) + control['conteos']['incidentes_seleccionados'] + control['conteos'].get('duplicados_exactos_por_llave', 0) == control['conteos']['filas_originales']
display(pd.Series(control['exclusiones_disjuntas'], name='filas'))
display(audit['dias_sin_registros_ciudad'])
daily = pd.read_parquet(backend / 'data/processed/serie_diaria.parquet')
weekly = pd.read_parquet(backend / 'data/processed/serie_semanal.parquet')
monthly = pd.read_parquet(backend / 'data/processed/serie_mensual.parquet')
for frame in [daily, weekly, monthly]:
    assert frame.conteo_observado_archivo.sum() == control['conteos']['incidentes_seleccionados']
display(weekly.loc[~weekly.apto_modelado_segun_cobertura].head(10))

## Revisar población y disponibilidad temporal
Solo se usó la hoja general y las filas de área Total, contrastadas contra sus áreas componentes. No se sumaron las hojas desagregadas por sexo o edad. La versión del archivo es posterior al histórico de accidentes: sus tasas son contexto retrospectivo y no una demostración de información disponible al pronosticar.

In [ ]:
population = pd.read_parquet(backend / 'data/processed/poblacion_comuna_anio.parquet')
assert len(population) == 80
assert not population.duplicated(['comuna_id', 'anio']).any()
assert population.poblacion.gt(0).all()
display(population.pivot(index='comuna_id', columns='anio', values='poblacion'))
display(daily.isna().sum())

## Patrones descriptivos del entrenamiento
Para cada gráfica explicar: unidad, periodo, denominador y limitaciones. Las barras son conteos, no riesgo individual. El mapa de calor divide por la cantidad de días de cada tipo presentes en el calendario con registros de ciudad. No inferir causalidad.

In [ ]:
for filename in ['tendencia_mensual.png', 'conteos_comuna.png', 'dia_franja.png']:
    display(Image(filename=str(reports / filename)))
display(pd.read_csv(reports / 'conteos_comuna_entrenamiento.csv'))

## Entrega a modelado
Target: accidentes registrados con víctimas por comuna y periodo. Filtrar cobertura y periodos completos; excluir semanas que cruzan particiones. Los rezagos y promedios móviles deberán usar exclusivamente pasado. No usar gravedad del incidente futuro como predictor.
Preguntas para defensa: ¿qué significa un cero?, ¿qué cambia al agregar por semana?, ¿cómo se evita doble conteo en población?, ¿por qué el test no guía la exploración?, ¿qué información estaba disponible al emitir el pronóstico?
Revisar las decisiones, los materiales de clase relacionados y los límites en `docs/DATOS.md`.